# Overture Maps Specific Source Extractor (Google vs. Microsoft)

This notebook allows you to query cloud building footprints within a specified bounding box, and filter specifically to extract **either Google Open Buildings or Microsoft ML Buildings** by changing a single variable in the configuration.

## How to use:
1. Modify the coordinates in **Section 1: Configuration** below.
2. Set the `selected_source` variable to "google", "microsoft", or "all".
3. Run all cells in this notebook.
4. Open the output CSV and GeoTIFF files directly in QGIS.

### Section 1: Configuration

In [7]:
# --- TARGET CONFIGURATION ---
# Bounding Box Coordinates (WGS84 Lat/Lon)
xmin = -69.8040  # Min Longitude (West)
ymin = 9.9660    # Min Latitude (South)
xmax = -69.7950  # Max Longitude (East)
ymax = 9.9750    # Max Latitude (North)

# Output File Paths
csv_out = "buildings_source.csv"
tif_out = "buildings.tif"

# Output Settings
resolution = 0.00001     # Raster resolution in degrees per pixel (default: 0.00001, ~1.1 meters)

# Source Selection:
# - "google"     : Extracts ONLY Google Open Buildings
# - "microsoft"  : Extracts ONLY Microsoft ML Buildings
# - "all"        : Extracts both Google and Microsoft buildings (including OSM additions)
selected_source = "google"

### Section 2: Imports & Helper Functions

In [8]:
import sys
import os
import ast
import json
import pandas as pd
import numpy as np
import rasterio
import shapely
from rasterio.transform import from_bounds
from rasterio.features import rasterize
import overturemaps

def get_max_update_time(sources_val):
    if not sources_val:
        return None
    try:
        if isinstance(sources_val, str):
            src_list = ast.literal_eval(sources_val)
        elif isinstance(sources_val, list):
            src_list = sources_val
        else:
            return None
        
        times = []
        for src in src_list:
            if isinstance(src, dict):
                ut = src.get('update_time')
                if ut:
                    times.append(ut[:10])
        if times:
            return max(times)
    except Exception:
        pass
    return None

### Section 3: Cloud Querying (Overture Live)

In [9]:
latest_release = overturemaps.core.get_latest_release()
data_release_version = f"Overture-{latest_release} (Live Cloud)"
print(f"Querying Overture Maps live via parallelized STAC index pruning (Release: {data_release_version})...")

bbox_tuple = (xmin, ymin, xmax, ymax)
reader = overturemaps.record_batch_reader("building", bbox=bbox_tuple, stac=True)
table = reader.read_all()
print(f"Fetched {len(table)} raw records from Overture.")

Querying Overture Maps live via parallelized STAC index pruning (Release: Overture-2026-06-17.0 (Live Cloud))...
Fetched 13 raw records from Overture.


### Section 4: Data Processing & Source Filtering

In [10]:
df_final = pd.DataFrame()
shapes_to_rasterize = []

if len(table) > 0:
    print("Vectorized parsing of WKB geometries...")
    wkb_bytes = table.column("geometry").to_pylist()
    geoms = shapely.from_wkb(wkb_bytes)
    
    ids = table.column("id").to_pylist()
    classes = table.column("class").to_pylist()
    heights = [float(h) if h is not None else 3.0 for h in table.column("height").to_pylist()]
    sources = table.column("sources").to_pylist()
    max_update_times = [get_max_update_time(s) for s in sources]
    
    df_raw = pd.DataFrame({
        "id": ids,
        "class": classes,
        "height": heights,
        "data_date": max_update_times,
        "sources": [str(s) for s in sources],
        "geometry_polygon_wkt": [g.wkt for g in geoms],
        "geom_obj": geoms
    })
    
    # Apply dataset source filter
    if selected_source == "google":
        print("Filtering for Google Open Buildings...")
        df_final = df_raw[df_raw["sources"].str.contains("Google Open Buildings", na=False)].copy()
    elif selected_source == "microsoft":
        print("Filtering for Microsoft ML Buildings...")
        df_final = df_raw[df_raw["sources"].str.contains("Microsoft ML Buildings", na=False)].copy()
    else:
        print("Retaining all sources...")
        df_final = df_raw.copy()
        
    # Standardize data capture date to a single latest date matching the user's requirement
    if not df_final.empty:
        df_final["data_date"] = df_final["data_date"].apply(lambda d: d[:10] if isinstance(d, str) else None)
        valid_dates = df_final["data_date"].dropna()
        if not valid_dates.empty:
            latest_date = valid_dates.max()
            print(f"Latest snapshot date for {selected_source}: {latest_date}")
            df_final = df_final[df_final["data_date"] == latest_date].copy()
        else:
            df_final = pd.DataFrame()
            
    if not df_final.empty:
        df_final["data_release_version"] = data_release_version
        shapes_to_rasterize = list(zip(df_final["geom_obj"], df_final["height"]))

print(f"Filtering complete. Kept {len(df_final)} buildings from {selected_source}.")

Vectorized parsing of WKB geometries...
Filtering for Google Open Buildings...
Latest snapshot date for google: 2023-05-01
Filtering complete. Kept 2 buildings from google.


### Section 5: CSV & GeoTIFF Export

In [11]:
if df_final.empty:
    cols = ["id", "class", "height", "data_date", "sources", "geometry_polygon_wkt", "data_release_version"]
    pd.DataFrame(columns=cols).to_csv(csv_out, index=False)
    print(f"No buildings matched for {selected_source}. Exported empty CSV: {csv_out}")
    if os.path.exists(tif_out):
        os.remove(tif_out)
else:
    # Save CSV
    out_cols = [c for c in df_final.columns if c != "geom_obj" and c != "geometry"]
    df_final[out_cols].to_csv(csv_out, index=False)
    print(f"Successfully saved CSV to: {csv_out}")
    
    # Save GeoTIFF
    print("Generating GeoTIFF raster...")
    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))
    
    transform = from_bounds(xmin, ymin, xmax, ymax, width, height)
    
    raster = rasterize(
        shapes_to_rasterize,
        out_shape=(height, width),
        transform=transform,
        fill=0.0,
        dtype="float32"
    )
    
    with rasterio.open(
        tif_out,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        nodata=0.0
    ) as dst:
        dst.write(raster, 1)
    print(f"Successfully saved GeoTIFF to: {tif_out}")
    print("Done! You can load these files directly in QGIS.")

Successfully saved CSV to: buildings_source.csv
Generating GeoTIFF raster...
Successfully saved GeoTIFF to: buildings.tif
Done! You can load these files directly in QGIS.
